# A2 · Phân đoạn ảnh với phép trộn đặc trưng theo biên

Nghiên cứu khảo sát ảnh hưởng của trọng số trộn dựa trên dự đoán biên đối với phân đoạn trimap Oxford-IIIT Pet. Hai biến thể sử dụng cùng ResNet18, dữ liệu, hàm mất mát và ngân sách huấn luyện. Chỉ cơ chế trộn đặc trưng khác nhau giữa hai biến thể.

Cơ chế trộn được chọn từ Bag trong bài *PIDNet: A Real-time Semantic Segmentation Network Inspired by PID Controllers* của Xu, Xiong và Bhattacharyya, IEEE/CVF CVPR 2023, tr. 19529–19539. [Bài báo và mã DOI](https://doi.org/10.1109/CVPR52729.2023.01871)

Notebook huấn luyện từng mô hình từ trạng thái khởi tạo, chọn checkpoint theo tập xác thực và đánh giá tập kiểm tra sau khi khóa đủ các lượt. Các bảng và hình được tính từ lượt chạy của notebook. Phạm vi hỗ trợ AI được ghi trong [AI_USE_REPORT.md](../AI_USE_REPORT.md).

## 1. Môi trường và cấu hình

Trên máy cục bộ, sử dụng kernel Python 3.12 đã cài PyTorch, torchvision và các gói trong `requirements.txt`. Trên Colab, mở notebook từ kho mã và chọn GPU hỗ trợ BF16 như L4 hoặc A100. Ô thiết lập nhận diện thư mục kho mã cục bộ hoặc tải mã nguồn vào `/content/CO5085-deep-learning` trong phiên Colab mới. Có thể chọn thư mục bản sao hiện có qua `CO5085_REPO_ROOT`.

Thiết lập chuẩn có hai biến thể × ba hạt giống × 12 epoch. `CO5085_DEVICE=cpu` chạy với FP32. `CO5085_SMOKE=1` chỉ kiểm tra luồng thực thi bằng tập con và một epoch, lưu bằng chứng riêng. Khi chạy lại với một giao thức mới, đặt `CO5085_OUTPUT_ROOT` sang thư mục khác. `CO5085_DATA_ROOT` chỉ định nơi giữ dữ liệu và `CO5085_A2_CACHE` chỉ định cache đã chuẩn bị.

In [ ]:
import os
from pathlib import Path
import subprocess
import sys

repo_url = "https://github.com/dathuynh1108/CO5085-deep-learning.git"
colab_root = Path("/content/CO5085-deep-learning")
if os.getenv("CO5085_REPO_ROOT"):
    repo = Path(os.environ["CO5085_REPO_ROOT"]).expanduser().resolve()
else:
    candidates = (Path.cwd(), *Path.cwd().parents, colab_root)
    repo = next((path for path in candidates if (path / "src/A2/run.py").is_file()), None)
    if repo is None and Path("/content").is_dir():
        subprocess.run(["git", "clone", "--depth", "1", repo_url, str(colab_root)], check=True)
        repo = colab_root
if repo is None or not (repo / "src/A2/run.py").is_file():
    raise RuntimeError("Đặt CO5085_REPO_ROOT tới thư mục chứa mã nguồn CO5085.")

os.chdir(repo)
sys.path.insert(0, str(repo / "src"))

In [ ]:
from copy import deepcopy
from io import BytesIO
import csv
import inspect
import json
from statistics import mean, stdev
import time

from IPython.display import Image, Markdown, display
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import numpy as np
import PIL
import torch
from torch.utils.data import Subset
import torchvision

from A2.data import IGNORE, prepare, read_config, sha256
from A2.models import PetSegmenter, segmentation_loss
from A2.run import evaluate, freeze, load_dataset, loader, normalize, run_paths, signature, test_run
from common.runtime import (
    atomic_csv, atomic_json, atomic_torch_save, choose_device, environment_info,
    json_hash, seed_everything, state_fingerprint, synchronize,
)

protocol = read_config()
config = deepcopy(protocol)
smoke = os.getenv("CO5085_SMOKE") == "1"
if smoke:
    config.update(epochs=1, seeds=[protocol["seeds"][0]])

data_root = Path(os.getenv("CO5085_DATA_ROOT", repo / "data")).expanduser().resolve()
cache = Path(os.getenv("CO5085_A2_CACHE", data_root / "a2_prepared")).expanduser().resolve()
default_output = repo / "runs/notebooks" / ("a2-smoke" if smoke else "a2")
output = Path(os.getenv("CO5085_OUTPUT_ROOT", default_output)).expanduser().resolve()
historical_root = (repo / "runs/assignment2").resolve()
if output.is_relative_to(historical_root) or historical_root.is_relative_to(output):
    raise ValueError("Chọn thư mục đầu ra riêng cho notebook A2.")

device = choose_device(os.getenv("CO5085_DEVICE", "auto"))
workers = int(os.getenv("CO5085_WORKERS", "0"))
threads = int(os.getenv("CO5085_THREADS", "2"))
if workers < 0 or threads < 1:
    raise ValueError("Số worker phải không âm và số luồng phải dương.")
if device.type == "cuda" and not torch.cuda.is_bf16_supported(including_emulation=False):
    raise RuntimeError("Cần GPU hỗ trợ BF16 hoặc đặt CO5085_DEVICE=cpu để dùng FP32.")
torch.set_num_threads(threads)
seed_everything(config["split_seed"])
precision = "bfloat16 autocast" if device.type == "cuda" else "float32"
print(json.dumps({"config": config, "device": str(device), "precision": precision,
                  "workers": workers, "smoke": smoke, "output": str(output)}, ensure_ascii=False, indent=2))

def show_figure(figure):
    with BytesIO() as buffer:
        figure.savefig(buffer, format="png", dpi=125, bbox_inches="tight")
        display(Image(data=buffer.getvalue()))
    plt.close(figure)

## 2. Dữ liệu và khảo sát nhãn

Oxford-IIIT Pet có 37 giống chó và mèo. Tập xác thực lấy 10 ảnh mỗi giống từ phân hoạch `trainval`, với hạt giống chia tập cố định. Phân hoạch kiểm tra theo danh sách chính thức sau khi loại ảnh trùng chính xác với tập trước. Hai archive được đối chiếu bằng SHA-256. Các thống kê và ảnh minh họa trong mục này lấy từ tập huấn luyện và xác thực. [Oxford-IIIT Pet, CC BY-SA 4.0](https://www.robots.ox.ac.uk/~vgg/data/pets/)

Ảnh và trimap được co theo cùng tỉ lệ về khung 192 × 192. Nội suy láng giềng gần nhất giữ các giá trị rời rạc của trimap. Mã 255 biểu diễn vùng đệm và được bỏ qua khi tính mất mát và chỉ số. Tập huấn luyện áp dụng lật ngang đồng thời ảnh và mặt nạ với xác suất 0,5.

In [ ]:
manifest_path = cache / "manifest.json"
if not manifest_path.is_file():
    if cache.exists():
        raise FileExistsError(f"Cache chuẩn bị dở. Chọn đường dẫn CO5085_A2_CACHE mới: {cache}")
    prepare(data_root, cache, protocol)
manifest = json.loads(manifest_path.read_text())
source_identity = signature(cache)
if manifest["config_sha256"] != source_identity["config_sha256"]:
    raise ValueError("Cache được chuẩn bị bằng cấu hình dữ liệu khác.")

train_full = load_dataset(cache, "train", manifest)
validation_full = load_dataset(cache, "validation", manifest)
for name, split in manifest["splits"].items():
    print(f"{name:<12} {split['size']:>5,} ảnh, {len(split['breed_counts'])} giống")
print(f"Số ảnh trùng khác phân hoạch đã loại: {len(manifest['excluded_cross_split_duplicates'])}")

In [ ]:
labels = ["Vật nuôi", "Nền", "Chuyển tiếp"]
partitions = ("train", "validation")
partition_labels = ("Huấn luyện", "Xác thực")
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5), constrained_layout=True)
for i, (partition, title) in enumerate(zip(partitions, partition_labels, strict=True)):
    split = manifest["splits"][partition]
    eda = split["eda"]
    counts = np.asarray(eda["valid_pixel_counts"])
    shares = 100 * counts / counts.sum()
    axes[0].bar(np.arange(3) + 0.34 * i, shares, width=0.34, label=title)
    q = eda["boundary_share_quartiles"]
    print(f"{title} · kích thước gốc trung vị {eda['width_quartiles'][2]:g} × {eda['height_quartiles'][2]:g} px")
    print(f"  Tỷ lệ dải chuyển tiếp Q1 / trung vị / Q3: {q[1]:.1%} / {q[2]:.1%} / {q[3]:.1%}")
    breed_counts = split["breed_counts"]
    axes[1].plot(range(1, 38), [breed_counts[str(n)] for n in range(1, 38)], label=title)
axes[0].set(xticks=np.arange(3) + 0.17, xticklabels=labels, ylabel="Tỷ lệ pixel hợp lệ (%)")
axes[1].set(xlabel="Mã giống", ylabel="Số ảnh")
for axis in axes:
    axis.legend()
show_figure(fig)

In [ ]:
mask_colors = ListedColormap(["#007d72", "#29364c", "#f3b341"])
mask_colors.set_bad("#e8edf2")
fig, axes = plt.subplots(2, 4, figsize=(12, 6), constrained_layout=True)
for column, index in enumerate(np.linspace(0, len(validation_full) - 1, 4, dtype=int)):
    axes[0, column].imshow(validation_full.images[index])
    axes[0, column].set_title(manifest["splits"]["validation"]["names"][index], fontsize=9)
    target = np.ma.masked_equal(validation_full.masks[index], IGNORE)
    axes[1, column].imshow(target, cmap=mask_colors, vmin=0, vmax=2, interpolation="nearest")
for axis in axes.ravel():
    axis.axis("off")
fig.supxlabel("Xanh lục: vật nuôi · Xanh đậm: nền · Vàng: chuyển tiếp · Xám nhạt: đệm")
show_figure(fig)

## 3. Mô hình, huấn luyện và xác thực

Với đặc trưng chi tiết $D$ và đặc trưng ngữ cảnh $C$, hai biến thể cùng tính $F = gD+(1-g)C$. Trọng số $g$ bằng $0{,}5$ ở biến thể cố định và bằng $\sigma(e)$ ở biến thể theo biên, với $e$ là logit của đầu dự đoán biên. Cả hai mô hình dùng tổng cross-entropy phân đoạn và BCE biên cân bằng, với hệ số $0{,}25$ cho BCE. Việc dùng cùng giám sát biên giúp tách tác động của cơ chế trộn trong thiết lập này.

Mỗi epoch gồm các bước huấn luyện AdamW và một lượt xác thực. Cosine annealing điều chỉnh tốc độ học của bộ mã hóa và các đầu dự đoán. Checkpoint được cập nhật khi mIoU xác thực tăng, do đó chọn epoch sớm nhất nếu bằng điểm.

In [ ]:
def train_one(variant, seed, path, identity, train_set, val_set):
    path.mkdir(parents=True, exist_ok=False)
    seed_everything(seed)
    model = PetSegmenter(variant, pretrained=True)
    initial_state = state_fingerprint(model.state_dict())
    model.to(device=device, memory_format=torch.channels_last)
    head_params = [param for name, param in model.named_parameters() if not name.startswith("encoder.")]
    optimizer = torch.optim.AdamW(
        [{"params": model.encoder.parameters(), "lr": config["encoder_lr"]},
         {"params": head_params, "lr": config["head_lr"]}],
        weight_decay=config["weight_decay"],
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=config["epochs"])
    train_batches = loader(train_set, config, workers, device, seed, shuffle=True)
    val_batches = loader(val_set, config, workers, device, seed)
    metadata = dict(identity, config=config, variant=variant, seed=seed, workers=workers,
                    initial_state_sha256=initial_state, params=sum(p.numel() for p in model.parameters()))
    atomic_json(path / "protocol.json", metadata)

    history = []
    best_score = -1.0
    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats(device)
    for epoch in range(1, config["epochs"] + 1):
        model.train()
        synchronize(device)
        started = time.perf_counter()
        loss_sum = torch.zeros((), device=device)
        for images, masks in train_batches:
            masks = masks.to(device, non_blocking=True, dtype=torch.long)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device.type, dtype=torch.bfloat16, enabled=device.type == "cuda"):
                logits, edges = model(normalize(images, device))
            loss = segmentation_loss(logits, edges, masks, config["boundary_weight"])
            loss.backward()
            optimizer.step()
            loss_sum += loss.detach() * len(images)

        synchronize(device)
        seconds = time.perf_counter() - started
        train_loss = float(loss_sum / len(train_set))
        validation, _ = evaluate(model, val_batches, device)
        if not np.isfinite([train_loss, validation["miou"]]).all():
            raise RuntimeError(f"Chỉ số không hữu hạn tại {path}, epoch {epoch}.")
        if validation["miou"] > best_score:
            best_score, best_epoch = validation["miou"], epoch
            atomic_torch_save(path / "best.pt", model.state_dict())
        history.append({
            "epoch": epoch, "train_loss": train_loss, "train_seconds": seconds,
            "val_miou": validation["miou"], "val_macro_dice": validation["macro_dice"],
            "val_transition_iou": validation["iou"][2],
            "encoder_lr": optimizer.param_groups[0]["lr"], "head_lr": optimizer.param_groups[1]["lr"],
        })
        atomic_csv(path / "history.csv", history)
        print(f"{variant:6} seed={seed} epoch={epoch:02d} loss={train_loss:.4f} "
              f"val_mIoU={validation['miou']:.4f} train={seconds:.1f}s", flush=True)
        scheduler.step()
    atomic_json(path / "run.json", dict(
        metadata, epochs=len(history), best_epoch=best_epoch, best_val_miou=best_score,
        total_train_seconds=sum(row["train_seconds"] for row in history),
        peak_allocated_mib=torch.cuda.max_memory_allocated(device) / 2**20 if device.type == "cuda" else None,
        checkpoint_sha256=sha256(path / "best.pt"), history_sha256=sha256(path / "history.csv"),
    ))

In [ ]:
def selected_indices(partition):
    total = manifest["splits"][partition]["size"]
    if not smoke:
        return range(total)
    limits = {"train": 32, "validation": 16, "test": 16}
    rng = np.random.default_rng(config["split_seed"])
    return np.sort(rng.choice(total, size=min(limits[partition], total), replace=False)).tolist()


def subset_for_run(dataset, partition):
    return Subset(dataset, selected_indices(partition)) if smoke else dataset


identity = dict(
    source_identity, notebook_config_sha256=json_hash(config), smoke=smoke,
    train_loop_sha256=json_hash({"source": inspect.getsource(train_one)}),
    sample_indices_sha256=json_hash({part: list(selected_indices(part)) for part in manifest["splits"]}),
    runtime=environment_info(device) | {"torchvision": torchvision.__version__, "numpy": np.__version__,
                                       "pillow": PIL.__version__, "precision": precision, "workers": workers},
)
train_set = subset_for_run(train_full, "train")
val_set = subset_for_run(validation_full, "validation")
lock_path = output / "frozen.json"
for variant, seed, run_path in run_paths(output, config):
    if (run_path / "run.json").is_file():
        record = json.loads((run_path / "run.json").read_text())
        if (any(record.get(key) != value for key, value in identity.items())
                or (record["variant"], record["seed"], record["epochs"]) != (variant, seed, config["epochs"])
                or record["checkpoint_sha256"] != sha256(run_path / "best.pt")
                or record["history_sha256"] != sha256(run_path / "history.csv")):
            raise ValueError(f"Bằng chứng lượt chạy không khớp giao thức notebook: {run_path}")
        print(f"Đã xác minh lượt hoàn tất: {variant}, seed {seed}")
        continue
    if lock_path.exists() or run_path.exists():
        raise FileExistsError(f"Lượt chạy dở hoặc giao thức đã khóa: {run_path}. Chọn CO5085_OUTPUT_ROOT mới.")
    train_one(variant, seed, run_path, identity, train_set, val_set)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
variant_colors = {"fixed": "#476b9a", "guided": "#b3532b"}
for variant, seed, run_path in run_paths(output, config):
    with (run_path / "history.csv").open() as stream:
        history = list(csv.DictReader(stream))
    epochs = [int(row["epoch"]) for row in history]
    style = ("-", "--", ":")[config["seeds"].index(seed)]
    line = dict(label=f"{variant}, seed {seed}", color=variant_colors[variant], linestyle=style,
                marker="o", markersize=2)
    axes[0].plot(epochs, [float(row["train_loss"]) for row in history], **line)
    axes[1].plot(epochs, [100 * float(row["val_miou"]) for row in history], **line)
axes[0].set(xlabel="Epoch", ylabel="Mất mát huấn luyện")
axes[1].set(xlabel="Epoch", ylabel="mIoU xác thực (%)")
axes[1].legend(fontsize=8)
fig.savefig(output / "learning_curves.png", dpi=150)
show_figure(fig)

## 4. Khóa giao thức và đánh giá tập kiểm tra

Tệp `frozen.json` xác nhận đủ lượt chạy, cùng trạng thái khởi tạo cho cặp biến thể ở mỗi hạt giống, và mã băm của các checkpoint đã chọn. Sau khi xem đường cong xác thực và chốt giao thức, nhập `TEST` tại ô xác nhận để mở tập kiểm tra. Phiên chạy tự động có thể truyền `CO5085_CONFIRM_TEST=1`, được kiểm tra sau khi tệp khóa hợp lệ.

Ma trận nhầm lẫn đếm pixel theo nhãn thực (hàng) và nhãn dự đoán (cột). IoU và Dice từng lớp được tính từ ma trận gộp toàn bộ ảnh của một lượt. mIoU và macro-Dice là trung bình đều theo lớp, nên dải chuyển tiếp có cùng trọng số với hai vùng còn lại.

In [ ]:
frozen = freeze(output, config, identity)
if lock_path.exists():
    if json.loads(lock_path.read_text()) != frozen:
        raise ValueError("Giao thức hoặc bằng chứng thay đổi sau khi khóa.")
else:
    atomic_json(lock_path, frozen)
print(f"Đã khóa {len(frozen['runs'])} lượt tại {lock_path}.")

In [ ]:
if os.getenv("CO5085_CONFIRM_TEST") != "1":
    if input("Nhập TEST để xác nhận đánh giá tập kiểm tra sau khi khóa giao thức: ").strip() != "TEST":
        raise RuntimeError("Chưa xác nhận đánh giá tập kiểm tra.")
if json.loads(lock_path.read_text()) != freeze(output, config, identity):
    raise ValueError("Giao thức hoặc checkpoint đã thay đổi.")

test_full = load_dataset(cache, "test", manifest)
test_set = subset_for_run(test_full, "test")
test_names = [manifest["splits"]["test"]["names"][i] for i in selected_indices("test")]
for variant, seed, run_path in run_paths(output, config):
    result_path = run_path / "test.json"
    if result_path.is_file():
        result = json.loads(result_path.read_text())
        if (result["checkpoint_sha256"] != sha256(run_path / "best.pt")
                or result["per_image_sha256"] != sha256(run_path / "test_per_image.csv")):
            raise ValueError(f"Bằng chứng kiểm tra đã thay đổi: {run_path}")
        print(f"Đã xác minh kết quả kiểm tra: {variant}, seed {seed}")
        continue
    test_run(variant, run_path, test_set, test_names, config, workers, device)

## 5. Chỉ số, ma trận nhầm lẫn và ảnh minh họa

Bảng trình bày kết quả từng hạt giống và trung bình ± độ lệch chuẩn mẫu theo biến thể. Hiệu mIoU theo cặp dùng cùng hạt giống. Ma trận gộp các hạt giống đã chạy thể hiện tỷ lệ lỗi theo lớp thực, với mỗi lượt có cùng tập ảnh kiểm tra. Ảnh minh họa sử dụng checkpoint của hạt giống nhỏ nhất và các vị trí đã xác định trong danh sách ảnh.

In [ ]:
results = {}
summary_rows = []
for variant, seed, run_path in run_paths(output, config):
    result = json.loads((run_path / "test.json").read_text())
    results[variant, seed] = result
    summary_rows.append(dict(
        variant=variant, seed=seed, samples=result["samples"], miou=result["miou"],
        macro_dice=result["macro_dice"], transition_iou=result["iou"][2],
    ))
atomic_csv(output / "metrics_by_seed.csv", summary_rows)
metrics = ("miou", "macro_dice", "transition_iou")
lines = ["| Biến thể | Hạt giống | mIoU (%) | Macro-Dice (%) | IoU chuyển tiếp (%) |", "|---|---:|---:|---:|---:|"]
for row in summary_rows:
    values = " | ".join(f"{100 * row[name]:.2f}" for name in metrics)
    lines.append(f"| {row['variant']} | {row['seed']} | {values} |")

aggregate = {}
for variant in config["variants"]:
    aggregate[variant] = {}
    for metric in metrics:
        values = [row[metric] for row in summary_rows if row["variant"] == variant]
        aggregate[variant][metric] = dict(mean=mean(values), sample_sd=stdev(values) if len(values) > 1 else None)
    values = [f"{100 * item['mean']:.2f}" + (f" ± {100 * item['sample_sd']:.2f}" if item["sample_sd"] is not None else "")
              for item in aggregate[variant].values()]
    lines.append(f"| {variant} | Tổng hợp | {' | '.join(values)} |")

deltas = {seed: results["guided", seed]["miou"] - results["fixed", seed]["miou"] for seed in config["seeds"]}
atomic_json(output / "summary.json", {"smoke": smoke, "variants": aggregate, "paired_miou_deltas": deltas})
display(Markdown("\n".join(lines)))
print("Chênh lệch mIoU theo cặp, tính bằng điểm phần trăm:",
      ", ".join(f"seed {seed}: {100 * delta:+.2f}" for seed, delta in deltas.items()))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4), layout="constrained")
confusions = {}
for variant, axis in zip(config["variants"], axes, strict=True):
    counts = np.asarray([results[variant, seed]["confusion"] for seed in config["seeds"]]).sum(axis=0)
    rates = 100 * counts / np.maximum(counts.sum(axis=1, keepdims=True), 1)
    confusions[variant] = {"counts": counts.tolist(), "row_percent": rates.tolist()}
    image = axis.imshow(rates, cmap="Blues", vmin=0, vmax=100)
    for (actual, predicted), value in np.ndenumerate(rates):
        axis.text(predicted, actual, f"{value:.1f}%", ha="center", va="center",
                  color="white" if value > 50 else "black")
    axis.set(title=variant, xlabel="Nhãn dự đoán", ylabel="Nhãn thực",
             xticks=range(3), xticklabels=labels, yticks=range(3), yticklabels=labels)
fig.colorbar(image, ax=axes, label="Tỷ lệ theo nhãn thực (%)")
atomic_json(output / "pixel_confusion.json", confusions)
fig.savefig(output / "pixel_confusion.png", dpi=150)
show_figure(fig)

In [ ]:
seed = min(config["seeds"])
positions = np.linspace(0, len(test_set) - 1, 4, dtype=int)
examples = [test_set[index] for index in positions]
images = torch.stack([item[0] for item in examples])
masks = torch.stack([item[1] for item in examples]).numpy()
fig, axes = plt.subplots(len(positions), 4, figsize=(11, 10), constrained_layout=True)
for row, position in enumerate(positions):
    axes[row, 0].imshow(images[row].permute(1, 2, 0))
    axes[row, 0].set_ylabel(test_names[position], fontsize=8)
    axes[row, 1].imshow(np.ma.masked_equal(masks[row], IGNORE), cmap=mask_colors, vmin=0, vmax=2)

for column, variant in enumerate(config["variants"], start=2):
    checkpoint = output / variant / f"seed_{seed}" / "best.pt"
    model = PetSegmenter(variant).to(device=device, memory_format=torch.channels_last)
    model.load_state_dict(torch.load(checkpoint, map_location=device, weights_only=True))
    model.eval()
    with torch.inference_mode(), torch.autocast(device.type, dtype=torch.bfloat16, enabled=device.type == "cuda"):
        logits, _ = model(normalize(images, device))
    predictions = logits.argmax(dim=1).cpu().numpy()
    for row, prediction in enumerate(predictions):
        axes[row, column].imshow(np.ma.masked_where(masks[row] == IGNORE, prediction),
                                 cmap=mask_colors, vmin=0, vmax=2)
    del model, logits
for column, title in enumerate(("Ảnh RGB", "Trimap thực", *config["variants"])):
    axes[0, column].set_title(title)
for axis in axes.ravel():
    axis.set(xticks=[], yticks=[])
fig.suptitle(f"Dự đoán từ checkpoint xác thực · seed {seed}")
fig.savefig(output / "predictions.png", dpi=150)
show_figure(fig)

## 6. Phạm vi đối chiếu với PIDNet

Thí nghiệm cô lập tác động của trọng số theo biên trong một bộ phân đoạn dùng ResNet18. PIDNet nghiên cứu đồng thời kiến trúc ba nhánh và các mô-đun tương tác trên dữ liệu cảnh đường phố. Chênh lệch mIoU ở đây được diễn giải trong phạm vi Oxford-IIIT Pet, cùng với IoU vùng chuyển tiếp và tỷ lệ nhầm vùng này sang vật nuôi hoặc nền. Dải chuyển tiếp của trimap là một vùng có bề rộng, nên kết quả phụ thuộc vào cách gán nhãn và độ phân giải ảnh.

Thư mục kết quả chứa cấu hình và môi trường (`protocol.json`), lịch sử từng epoch (`history.csv`), checkpoint đã chọn (`best.pt`), tệp khóa (`frozen.json`), chỉ số từng ảnh và các bảng tổng hợp. Báo cáo A2 trình bày thêm phương pháp, thí nghiệm gốc và giới hạn khi chuyển cơ chế Bag sang bài toán này. [Báo cáo A2](../reports/assignment2/assignment2.pdf)